# Desafio Flor de Íris (Parte 1) — Geometria Multiclasse e `Perceptron`

Neste desafio, queremos classificar flores em **3 espécies** (`Iris-setosa`, `Iris-versicolor` e `Iris-virginica`) usando quatro medidas geométricas em `../arquivos-csv/iris_train.csv`:
- `sepal_length`, `sepal_width`, `petal_length`, `petal_width`

## 1. Inspeção Visual Antes de Modelar
Como engenheiros, nunca treinamos um classificador no escuro! Execute a célula abaixo para enxergar como as 3 espécies se distribuem no plano `(petal_length, petal_width)`:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# Carrega os dados rotulados de treinamento
iris_train = pd.read_csv("../arquivos-csv/iris_train.csv")

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=iris_train,
    x="petal_length",
    y="petal_width",
    hue="class",
    style="class",
    s=80,
    palette="deep"
)
plt.title("Geometria das Pétalas por Espécie de Íris")
plt.xlabel("Comprimento da Pétala (cm)")
plt.ylabel("Largura da Pétala (cm)")
plt.tight_layout()
plt.show()

**O que o gráfico revela?**
- A **`Iris-setosa`** (canto inferior esquerdo) é **100% linearmente separável** das outras duas com uma simples régua!
- Já **`Iris-versicolor`** e **`Iris-virginica`** encostam uma na outra na região central, criando uma fronteira delicada onde um classificador puramente linear pode cometer alguns erros.

---

## 2. Como um Perceptron Faz Classificação de 3 Classes? (*One-vs-Rest*)
Um neurônio Perceptron decide apenas entre dois lados (`Sim` vs. `Não`). Para classificar **3 classes**, treinamos **3 neurônios em paralelo** (estratégia **Um-contra-Todos / *One-vs-Rest***):
- Neurônio 0: *"É `Iris-setosa` (`1`) ou outra qualquer (`0`)?"*
- Neurônio 1: *"É `Iris-versicolor` (`1`) ou outra qualquer (`0`)?"*
- Neurônio 2: *"É `Iris-virginica` (`1`) ou outra qualquer (`0`)?"*
Na hora de prever, vence o neurônio que apresentar o **maior potencial de ativação $u$** (`np.argmax`)!

## O Desafio (Parte 1)
Treine um modelo linear **Perceptron** em `../arquivos-csv/iris_train.csv`, aplique sobre `../arquivos-csv/iris_exercise.csv` e salve um CSV com colunas `id,class`. Tente na célula abaixo antes de abrir os gabaritos!

In [ ]:
# Escreva sua implementação aqui!

<details>
<summary><b>Clique aqui para ver o Gabarito 1.1 — Perceptron Multiclasse com Scikit-Learn (Comentado)</b></summary>

```python
import pandas as pd
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler

# 1. Carrega treino (com coluna 'class') e exercício (sem coluna 'class')
train = pd.read_csv("../arquivos-csv/iris_train.csv")
test = pd.read_csv("../arquivos-csv/iris_exercise.csv")

colunas_entrada = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train = train[colunas_entrada]
y_train = train["class"]
X_test = test[colunas_entrada]

# 2. Padronização Z-score calibrada exclusivamente no conjunto de treino
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. O Perceptron do Scikit-Learn aplica automaticamente One-vs-Rest para 3 classes!
clf = Perceptron(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)
print(f"Acurácia no próprio conjunto de treino: {clf.score(X_train_scaled, y_train) * 100:.2f}%")

# 4. Previsão no conjunto de exercício e salvamento do CSV
y_pred = clf.predict(X_test_scaled)
pd.DataFrame({"id": test["id"], "class": y_pred}).to_csv("solution_iris_perceptron_sklearn.csv", index=False)
print("Arquivo gerado: solution_iris_perceptron_sklearn.csv")
```

</details>

<details>
<summary><b>Clique aqui para ver o Gabarito 1.2 — Perceptron Multiclasse em Python/NumPy Puro (Comentado)</b></summary>

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("../arquivos-csv/iris_train.csv")
test = pd.read_csv("../arquivos-csv/iris_exercise.csv")

colunas_entrada = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train_raw = train[colunas_entrada].to_numpy()
y_train_raw = train["class"].to_numpy()
X_test_raw = test[colunas_entrada].to_numpy()

# 1. Lista ordenada das 3 espécies e conversão para índices 0, 1, 2
classes = list(np.unique(y_train_raw))
y_train_idx = np.array([classes.index(nome) for nome in y_train_raw])

# 2. Padronização Z-score (média 0, desvio 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 3. Matriz de pesos: 3 neurônios (linhas) x 5 conexões (bias x0=-1 + 4 atributos)
pesos = np.zeros((len(classes), X_train.shape[1] + 1))
lr = 0.01
epocas = 150

for _ in range(epocas):
    for i in range(len(X_train)):
        x = np.insert(X_train[i], 0, -1.0)  # Vetor de entrada com x0 = -1 na frente
        # Treina cada um dos 3 neurônios no esquema Um-contra-Todos (One-vs-Rest)
        for c_idx in range(len(classes)):
            alvo = 1 if y_train_idx[i] == c_idx else 0
            y_hat = 1 if np.dot(pesos[c_idx], x) >= 0.0 else 0
            pesos[c_idx] += lr * (alvo - y_hat) * x

# 4. Na hora de prever, escolhemos a classe cujo neurônio teve MAIOR potencial u!
previsoes = []
for x_row in X_test:
    x_com_bias = np.insert(x_row, 0, -1.0)
    potenciais = [np.dot(pesos[c], x_com_bias) for c in range(len(classes))]
    vencedor = int(np.argmax(potenciais))
    previsoes.append(classes[vencedor])

pd.DataFrame({"id": test["id"], "class": previsoes}).to_csv("solution_iris_perceptron_puro.csv", index=False)
print("Arquivo gerado: solution_iris_perceptron_puro.csv")
```

</details>

---

## Próximo Passo
Para contornar a região não linear entre *Iris-versicolor* e *Iris-virginica* com alta precisão, vamos construir uma Rede Neural Multicamadas com **One-Hot Encoding** e **Backpropagation** na **[Parte 2 — Classificando Íris com MLP](02_desafio_iris_parte2_mlp.ipynb)**!